In [0]:
from functools import reduce
from itertools import chain

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def normalize_city_name(city: str):
    """
    Normalize a city name for prefix-based matching.

    The normalization:
    - trims surrounding whitespace
    - converts text to lowercase
    - replaces Finnish/Scandinavian characters with ASCII equivalents

    Args:
        city: Raw city name string.

    Returns:
        A normalized city name string.
    """
    return (
        city.strip()
        .lower()
        .replace("ä", "a")
        .replace("ö", "o")
        .replace("å", "a")
    )

def prepare_weather_inference_data(raw_table: str):
    """
    Load and prepare raw weather inference data from the bronze layer.

    This function:
    - reads the raw inference table from `fortum_challenge_data.01_bronze`
    - normalizes city names for prefix-based mapping
    - maps each city to a `weather_key`
    - converts temperature to double
    - casts the raw timestamp column to `timestamp_utc`
    - drops unused raw columns
    - removes the latest available row for each `weather_key`
    - returns the result ordered by `weather_key` and `timestamp_utc`

    The latest row per `weather_key` is removed so that the downstream
    inference dataset aligns with the intended forecasting setup.

    Args:
        raw_table: Name of the bronze-layer raw inference table.

    Returns:
        A Spark DataFrame containing cleaned inference weather data with
        at least:
        - `timestamp_utc`
        - `temperature`
        - `weather_key`
    """

    # City prefixes (first 3 letters).
    city_prefixes = ["HEL", "TUR", "TAM", "JOE", "JYV", "KUO", "LAH", "LAP", "OUL", "ROV", "VAA", "HAM", "ESP", "SEI", "VAN", "POR"]

    df = spark.table(f"fortum_challenge_data.01_bronze.{raw_table}")

    cities = [row["city"] for row in df.select("city").distinct().collect()]
    cities_clean = [(c, normalize_city_name(c)) for c in cities]

    prefix_to_cities = {
        p: [clean for (_, clean) in cities_clean if clean.upper().startswith(p)]
        for p in city_prefixes
    }

    city_to_prefix = {
        city_clean: prefix
        for prefix, matched_cities in prefix_to_cities.items()
        for city_clean in matched_cities
    }

    mapping_expr = F.create_map([F.lit(x) for x in chain(*city_to_prefix.items())])

    weather_long = (
        df
        .withColumn(
            "city_clean",
            F.lower(F.trim(F.expr("replace(replace(replace(city, 'ä', 'a'), 'Ä', 'A'), 'ö', 'o')")))
        )
        .drop("PRA_PT1H_ACC", "WS_PT1H_AVG")
        .withColumn("temperature", F.expr("try_cast(`TA_PT1H_AVG` as double)"))
        .withColumn("timestamp_utc", F.col("timestamp").cast("timestamp"))
        .withColumn("weather_key", mapping_expr[F.col("city_clean")])
        .drop("TA_PT1H_AVG", "city", "city_clean", "timestamp")
    )
    # Window that orders rows from latest to earliest within each city
    window_desc = Window.partitionBy("weather_key").orderBy(F.col("timestamp_utc").desc())

    weather_24h = (
        weather_long
        .withColumn("rn_desc", F.row_number().over(window_desc))
        .filter(F.col("rn_desc") > 1)   # remove the latest row of each weather_key
        .drop("rn_desc")
    )

    weather_asc = weather_24h.orderBy("weather_key", "timestamp_utc")
    return weather_asc


def prepare_weather_training_data(raw_table: str):
    """
    Load and prepare raw historical weather training data from the bronze layer.

    This function:
    - reads the raw training table from `fortum_challenge_data.01_bronze`
    - builds a proper timestamp column named `timestamp_utc` from
      year/month/day/hour components
    - drops date-part columns that are no longer needed

    Args:
        raw_table: Name of the bronze-layer raw training table.

    Returns:
        A Spark DataFrame containing cleaned historical weather data with
        a `timestamp_utc` column.
    """

    df = spark.table(f"fortum_challenge_data.01_bronze.{raw_table}")

    df_clean = df.withColumn(
        "timestamp_utc",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.concat_ws("-", F.col("Vuosi"), F.col("Kuukausi"), F.col("Päivä")),
                F.col("hour")
            )
        )
    )

    # Drop the original date-part columns since timestamp_utc now contains the same information
    # Havaintoasema is also dropped here because it is not needed for the next steps
    df_clean = df_clean.drop("Vuosi", "Kuukausi", "Päivä", "hour", "Havaintoasema")

    return df_clean


def build_complete_weather_features(df: DataFrame, min_ts, max_ts, min_history_rows: int = 336):
    """
    Build a complete hourly weather feature dataset for each weather location.

    This function:
    - generates a full hourly timestamp spine between `min_ts` and `max_ts`
    - creates a full grid of `weather_key` x hour
    - left joins the observed weather data onto that grid
    - marks missing temperature observations
    - forward-fills missing temperature values within each `weather_key`
    - creates lag features from the filled temperature series
    - drops raw temperature columns
    - removes the earliest rows that do not have enough historical context

    Lag features created:
    - `temp_lag_24`
    - `temp_lag_48`
    - `temp_lag_168`

    Args:
        df: Spark DataFrame containing at least `weather_key`,
            `timestamp_utc`, and `temperature`.
        min_ts: Inclusive lower bound of the hourly time spine.
        max_ts: Inclusive upper bound of the hourly time spine.
        min_history_rows: Number of earliest rows to remove per `weather_key`
            to ensure sufficient history for lag-based features. Defaults to 336.

    Returns:
        A Spark DataFrame containing completed hourly weather features.
    """

    spine = (spark.sql(f"""
    SELECT explode(sequence(
        timestamp('{min_ts}'),
        timestamp('{max_ts}'),
        interval 1 hour
    )) AS timestamp_utc
    """))

    weather_keys = df.select("weather_key").distinct()
    grid = weather_keys.crossJoin(spine)

    complete = (grid
        .join(df, on=["weather_key", "timestamp_utc"], how="left")
    )

    window = Window.partitionBy("weather_key").orderBy("timestamp_utc")

    complete_sorted = complete.orderBy("weather_key", "timestamp_utc")

    weather_with_missing_flag  = complete_sorted.withColumn(
        "temperature_missing_flag",
        F.when(F.col("temperature").isNull(), 1).otherwise(0)
    )

    # Forward-fill missing temperature values using the most recent non-null value
    # within each weather_key partition.
    weather_filled = weather_with_missing_flag.withColumn("temperature_filled", F.last("temperature", ignorenulls=True).over(window))

    weather_with_lags = weather_filled.withColumn("temp_lag_24", F.lag("temperature_filled", 24).over(window)) \
            .withColumn("temp_lag_48", F.lag("temperature_filled", 48).over(window)) \
            .withColumn("temp_lag_168", F.lag("temperature_filled", 168).over(window)) 

    weather_with_dropped = weather_with_lags.drop("temperature", "temperature_filled")


    # Add row index per weather_key
    weather_with_row_num = weather_with_dropped.withColumn(
        "row_num",
        F.row_number().over(window)
    )

    # Drop first 336 rows
    weather_final = weather_with_row_num.filter(F.col("row_num") > min_history_rows).drop("row_num")

    return weather_final


def drop_latest_forecast_horizon(df: DataFrame, horizon_hours: int = 48):
    """
    Drop the most recent rows per `weather_key`.

    This is useful when the latest observations should be excluded from the
    final training dataset so that training and inference periods remain
    properly separated.

    Args:
        df: Spark DataFrame containing `weather_key` and `timestamp_utc`.
        horizon_hours: Number of most recent rows to remove per `weather_key`.
            Defaults to 48.

    Returns:
        A Spark DataFrame ordered by `weather_key` and `timestamp_utc`
        with the latest `horizon_hours` rows removed per `weather_key`.
    """

    # Window that orders rows from latest to earliest within each city
    window_desc = Window.partitionBy("weather_key").orderBy(F.col("timestamp_utc").desc())

    trimmed_df = (
        df
        .withColumn("rn_desc", F.row_number().over(window_desc))
        .filter(F.col("rn_desc") > horizon_hours)   # remove the latest row of each weather_key
        .drop("rn_desc")
    )

    weather_asc = trimmed_df.orderBy("weather_key", "timestamp_utc")

    return weather_asc


def write_weather_table(df: DataFrame, table_name: str):
    """
    Write a weather feature DataFrame to the silver layer.

    Args:
        df: Spark DataFrame to write.
        table_name: Destination table name within
            `fortum_challenge_data.02_silver`.

    Returns:
        None.

    Side Effects:
        Overwrites the destination silver table.
    """
    df.write \
        .mode("overwrite") \
        .saveAsTable(f"fortum_challenge_data.02_silver.{table_name}")
    print(f"Successfully wrote to silver table {table_name}")


if __name__ == "__main__":
    raw_inference_table = "weather_hourly_inference_raw"
    raw_training_table = "weather_data_long"

    clean_training_table = "weather_training_clean"
    clean_inference_table = "weather_inference_clean"
    clean_inference_null_table = "weather_inference_null_clean"

    inf_min_ts = "2024-09-15T00:00:00.000+00:00"
    inf_min_ts_null = "2024-09-22T00:00:00.000+00:00"
    inf_max_ts = "2024-09-30T23:00:00.000+00:00"
    train_min_ts = "2021-01-01T00:00:00.000+00:00"
    train_max_ts = "2024-09-30T23:00:00.000+00:00"

    inference_weather_raw = prepare_weather_inference_data(raw_inference_table)
    training_weather_raw = prepare_weather_training_data(raw_training_table)

    inference_weather_features = build_complete_weather_features(inference_weather_raw, inf_min_ts, inf_max_ts)
    inference_weather_features_null = build_complete_weather_features(inference_weather_raw, inf_min_ts_null, inf_max_ts)
    training_weather_features = build_complete_weather_features(training_weather_raw, train_min_ts, train_max_ts)
    training_weather_final = drop_latest_forecast_horizon(training_weather_features)

    write_weather_table(training_weather_final, clean_training_table)
    write_weather_table(inference_weather_features, clean_inference_table)
    write_weather_table(inference_weather_features_null, clean_inference_null_table)

    inference_weather_features.display()